# Sentiment140: Seven Text Classification Experiments

This notebook compares seven approaches for binary sentiment classification. It includes dataset inspection, text cleaning, a stratified 70/30 train-test split, a 20% validation split from the training portion, and test-set Accuracy, Precision, Recall and F1 for every experiment.

**Dataset note:** Sentiment140 contains labels `0` (negative) and `4` (positive), but no `1` (neutral) examples. The notebook reports all three requested counts and maps `4 → 1` only for binary model training. Therefore, this is a negative-vs-positive task, not a three-class task.

Set `SAMPLE_SIZE = None` to use all 1.6 million tweets. The default sample makes seven model runs more practical; in that case, the test set is 30% of the sampled dataset.

## 0. Install and import dependencies

In [4]:
%pip -q install kagglehub nltk scikit-learn tensorflow pandas numpy

In [5]:
import os
import re
import string
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

SEED = 1337
MAX_TOKENS = 10_000
MAX_LENGTH = 140
BATCH_SIZE = 256
EPOCHS = 5
SAMPLE_SIZE = 100_000  # Set to None to use all rows

np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


## 1. Load the dataset and inspect class counts

In [6]:
import kagglehub

data_dir = kagglehub.dataset_download("kazanova/sentiment140")
csv_path = os.path.join(data_dir, "training.1600000.processed.noemoticon.csv")
column_names = ["target", "tweet_id", "date", "query", "user", "text"]
df = pd.read_csv(csv_path, encoding="latin-1", names=column_names, usecols=[0, 5])

class_counts = df["target"].value_counts().reindex([0, 1, 4], fill_value=0)
class_counts.index = ["0 - negative", "1 - neutral", "4 - positive"]
display(class_counts.to_frame("samples"))
print("Neutral examples are absent from the standard Sentiment140 dataset.")

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    # Sample equally by original label so both classes remain represented.
    df = (df.groupby("target", group_keys=False)
            .apply(lambda part: part.sample(n=max(1, int(SAMPLE_SIZE * len(part) / len(df))), random_state=SEED))
            .reset_index(drop=True))
    if len(df) > SAMPLE_SIZE:
        df = df.sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

# Convert original 0/4 labels to binary 0/1 for sigmoid output.
df["label"] = df["target"].map({0: 0, 4: 1}).astype("int32")
print(f"Rows used for experiments: {len(df):,}")
display(df.head())

Using Colab cache for faster access to the 'sentiment140' dataset.


,samples
0 - negative,800000
1 - neutral,0
4 - positive,800000


Neutral examples are absent from the standard Sentiment140 dataset.
Rows used for experiments: 100,000


/tmp/ipykernel_1593/1877248569.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda part: part.sample(n=max(1, int(SAMPLE_SIZE * len(part) / len(df))), random_state=SEED))


,target,text,label
0,0,@girltarist yes mam! No wristband though are ...,0
1,0,gettn ready to take a trip to Jersey my dad's ...,0
2,0,@redrawsounds He is good and might be useful f...,0
3,0,pre-ordered the new 32 GB iPhone today! Can't...,0
4,0,@RuggyYelpSD just watched the price is right v...,0


## 2. Clean text: lowercase, remove punctuation and English stop-words

In [7]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
# Keep negation terms because removing words such as "not" can reverse sentiment.
stop_words -= {"no", "nor", "not", "never"}
punctuation_table = str.maketrans("", "", string.punctuation)

def clean_tweet(value):
    value = str(value).lower()
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    value = value.translate(punctuation_table)
    tokens = [word for word in value.split() if word not in stop_words]
    return " ".join(tokens)

df["clean_text"] = df["text"].map(clean_tweet)
df = df.loc[df["clean_text"].str.strip().ne("")].reset_index(drop=True)
display(df[["text", "clean_text", "target", "label"]].head())

,text,clean_text,target,label
0,@girltarist yes mam! No wristband though are ...,girltarist yes mam no wristband though,0,0
1,gettn ready to take a trip to Jersey my dad's ...,gettn ready take trip jersey dads not good nee...,0,0
2,@redrawsounds He is good and might be useful f...,redrawsounds good might useful us unfortunatel...,0,0
3,pre-ordered the new 32 GB iPhone today! Can't...,preordered new 32 gb iphone today cant waitonl...,0,0
4,@RuggyYelpSD just watched the price is right v...,ruggyyelpsd watched price right video ridiculo...,0,0


## 3. Stratified 70/30 train-test split and 20% validation split

In [8]:
X = df["clean_text"].to_numpy(dtype=str)
y = df["label"].to_numpy(dtype=np.int32)

X_train_all, X_test, y_train_all, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all, test_size=0.20,
    stratify=y_train_all, random_state=SEED
)

print(f"Training:   {len(X_train):,} ({len(X_train)/len(X):.1%} of full working set)")
print(f"Validation: {len(X_val):,} ({len(X_val)/len(X):.1%} of full working set)")
print(f"Test:       {len(X_test):,} ({len(X_test)/len(X):.1%} of full working set)")
print("Train label proportions:", np.bincount(y_train) / len(y_train))
print("Test label proportions: ", np.bincount(y_test) / len(y_test))

def make_text_dataset(texts, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(min(len(texts), 20_000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_text_dataset(X_train, y_train, shuffle=True)
val_ds = make_text_dataset(X_val, y_val)
test_ds = make_text_dataset(X_test, y_test)
text_only_train = tf.data.Dataset.from_tensor_slices(X_train).batch(BATCH_SIZE)

Training:   55,986 (56.0% of full working set)
Validation: 13,997 (14.0% of full working set)
Test:       29,993 (30.0% of full working set)
Train label proportions: [0.50003572 0.49996428]
Test label proportions:  [0.50005001 0.49994999]


## Shared helpers

In [9]:
results = []

def build_dense_model(input_dim):
    inputs = keras.Input(shape=(input_dim,), dtype=tf.float32)
    x = layers.Dense(32, activation="relu")(inputs)
    x = layers.Dense(16, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def evaluate_model(model, test_features, name):
    """Show all tables for ONE model (call this in that model's own cell)."""
    from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
    probabilities = model.predict(test_features, verbose=0).reshape(-1)
    predictions = (probabilities >= 0.5).astype("int32")
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
    }
    # keep one row per model for the final comparison (safe to re-run this cell)
    results[:] = [r for r in results if r["Model"] != name]
    results.append(row)

    print("=" * 70)
    print(f"RESULTS: {name}")
    print("=" * 70)

    # Table 1: Keras model summary (Layer (type) | Output Shape | Param #)
    print("\nTable 1 - Model summary")
    model.summary()

    # Table 2: overall test-set metrics
    print("\nTable 2 - Test metrics")
    display(pd.DataFrame([row]).set_index("Model").round(4))

    # Table 3: per-class metrics
    p, r, f, s = precision_recall_fscore_support(y_test, predictions, labels=[0, 1], zero_division=0)
    per_class = pd.DataFrame(
        {"Precision": p, "Recall": r, "F1": f, "Support": s},
        index=["Negative (0)", "Positive (1)"],
    )
    print("\nTable 3 - Per-class metrics")
    display(per_class.round(4))

    # Table 4: confusion matrix
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=["Actual Negative", "Actual Positive"],
        columns=["Predicted Negative", "Predicted Positive"],
    )
    print("\nTable 4 - Confusion matrix")
    display(cm_df)
    return row

def vectorize_datasets(vectorizer):
    def transform(texts, labels):
        return vectorizer(texts), labels
    tr = train_ds.map(transform, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    va = val_ds.map(transform, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    te = test_ds.map(transform, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    return tr, va, te

def fit_dense_experiment(ngrams, mode):
    vectorizer = TextVectorization(max_tokens=MAX_TOKENS, ngrams=ngrams, output_mode=mode)
    vectorizer.adapt(text_only_train)
    tr, va, te = vectorize_datasets(vectorizer)
    model = build_dense_model(len(vectorizer.get_vocabulary()))
    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
    model.fit(tr, validation_data=va, epochs=EPOCHS, callbacks=callbacks, verbose=1)
    return model, vectorizer, te

## Experiment 1 — unigram `multi_hot` (fully connected 32 → 16)

In [10]:
model_1, vectorizer_1, test_1 = fit_dense_experiment(ngrams=None, mode="multi_hot")

Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.7271 - loss: 0.5628 - val_accuracy: 0.7657 - val_loss: 0.4962
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.7988 - loss: 0.4566 - val_accuracy: 0.7640 - val_loss: 0.4923
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - accuracy: 0.8234 - loss: 0.4106 - val_accuracy: 0.7624 - val_loss: 0.5040
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.8440 - loss: 0.3678 - val_accuracy: 0.7582 - val_loss: 0.5269


### Experiment 1 — evaluation tables

In [11]:
evaluate_model(model_1, test_1, "1. Unigram multi_hot")

RESULTS: 1. Unigram multi_hot

Table 1 - Model summary


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       320,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 961,733 (3.67 MB)

 Trainable params: 320,577 (1.22 MB)

...


Table 3 - Per-class metrics


,Precision,Recall,F1,Support
Negative (0),0.7731,0.7553,0.7641,14998
Positive (1),0.7608,0.7783,0.7694,14995



Table 4 - Confusion matrix


,Predicted Negative,Predicted Positive
Actual Negative,11328,3670
Actual Positive,3324,11671


{'Model': '1. Unigram multi_hot',
 'Accuracy': 0.7668122561931118,
 'Precision': 0.760771788019034,
 'Recall': 0.778326108702901,
 'F1': 0.7694488396624473}

## Experiment 2 — unigram + bigram `multi_hot`

In [12]:
model_2, vectorizer_2, test_2 = fit_dense_experiment(ngrams=2, mode="multi_hot")

Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 8s 31ms/step - accuracy: 0.7196 - loss: 0.5743 - val_accuracy: 0.7717 - val_loss: 0.4932
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.7991 - loss: 0.4581 - val_accuracy: 0.7714 - val_loss: 0.4857
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.8223 - loss: 0.4118 - val_accuracy: 0.7688 - val_loss: 0.4944
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.8395 - loss: 0.3749 - val_accuracy: 0.7660 - val_loss: 0.5108


### Experiment 2 — evaluation tables

In [13]:
evaluate_model(model_2, test_2, "2. Bigram multi_hot")

RESULTS: 2. Bigram multi_hot

Table 1 - Model summary


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │       320,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 961,733 (3.67 MB)

 Trainable params: 320,577 (1.22 MB)

...


Table 3 - Per-class metrics


,Precision,Recall,F1,Support
Negative (0),0.7804,0.7501,0.7649,14998
Positive (1),0.7594,0.7889,0.7738,14995



Table 4 - Confusion matrix


,Predicted Negative,Predicted Positive
Actual Negative,11250,3748
Actual Positive,3166,11829


{'Model': '2. Bigram multi_hot',
 'Accuracy': 0.7694795452272197,
 'Precision': 0.7593888425242344,
 'Recall': 0.788862954318106,
 'F1': 0.7738453486850713}

## Experiment 3 — unigram + bigram `tf_idf`

In [14]:
model_3, vectorizer_3, test_3 = fit_dense_experiment(ngrams=2, mode="tf_idf")

Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.6964 - loss: 0.5886 - val_accuracy: 0.7670 - val_loss: 0.4971
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - accuracy: 0.7988 - loss: 0.4505 - val_accuracy: 0.7687 - val_loss: 0.4973
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 7s 34ms/step - accuracy: 0.8276 - loss: 0.3890 - val_accuracy: 0.7677 - val_loss: 0.5133


### Experiment 3 — evaluation tables

In [15]:
evaluate_model(model_3, test_3, "3. Bigram tf_idf")

RESULTS: 3. Bigram tf_idf

Table 1 - Model summary


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │       320,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 961,733 (3.67 MB)

 Trainable params: 320,577 (1.22 MB)

...


Table 3 - Per-class metrics


,Precision,Recall,F1,Support
Negative (0),0.7727,0.7535,0.7630,14998
Positive (1),0.7594,0.7783,0.7688,14995



Table 4 - Confusion matrix


,Predicted Negative,Predicted Positive
Actual Negative,11301,3697
Actual Positive,3324,11671


{'Model': '3. Bigram tf_idf',
 'Accuracy': 0.7659120461441002,
 'Precision': 0.7594351900052057,
 'Recall': 0.778326108702901,
 'F1': 0.7687646148272569}

## Prepare integer sequences for Experiments 4–7

In [16]:
sequence_vectorizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=MAX_LENGTH,
)
sequence_vectorizer.adapt(text_only_train)
sequence_train, sequence_val, sequence_test = vectorize_datasets(sequence_vectorizer)
actual_vocab_size = len(sequence_vectorizer.get_vocabulary())
print("Vocabulary size:", actual_vocab_size, "| sequence length:", MAX_LENGTH)

Vocabulary size: 10000 | sequence length: 140


## Experiment 4 — trainable Embedding + LSTM (32 units)

In [17]:
def build_lstm_model(embedding_matrix=None, embedding_dim=128, trainable=True):
    inputs = keras.Input(shape=(MAX_LENGTH,), dtype="int64")
    if embedding_matrix is None:
        embedded = layers.Embedding(
            input_dim=actual_vocab_size, output_dim=embedding_dim,
            mask_zero=True, name="learned_embedding"
        )(inputs)
    else:
        embedded = layers.Embedding(
            input_dim=embedding_matrix.shape[0], output_dim=embedding_matrix.shape[1],
            embeddings_initializer=keras.initializers.Constant(embedding_matrix),
            trainable=trainable, mask_zero=True, name="pretrained_embedding"
        )(inputs)
    x = layers.LSTM(32)(embedded)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

model_4 = build_lstm_model(embedding_dim=128)
stop_early = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
model_4.fit(sequence_train, validation_data=sequence_val, epochs=EPOCHS, callbacks=stop_early, verbose=1)


Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 81s 360ms/step - accuracy: 0.7348 - loss: 0.5429 - val_accuracy: 0.7721 - val_loss: 0.4861
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 85s 386ms/step - accuracy: 0.8008 - loss: 0.4427 - val_accuracy: 0.7704 - val_loss: 0.4865
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 77s 351ms/step - accuracy: 0.8200 - loss: 0.4076 - val_accuracy: 0.7683 - val_loss: 0.4987


### Experiment 4 — evaluation tables

In [18]:
evaluate_model(model_4, sequence_test, "4. Learned embedding + LSTM")

RESULTS: 4. Learned embedding + LSTM

Table 1 - Model summary


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 140)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ learned_embedding   │ (None, 140, 128)  │  1,280,000 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 140)       │          0 │ input_layer_3[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 32)        │     20,608 │ learned_embeddin… │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 1)         │         33 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,901,925 (14.88 MB)

 Trainable params: 1,300,641 (4.96 MB)

...


Table 3 - Per-class metrics


,Precision,Recall,F1,Support
Negative (0),0.7631,0.7824,0.7726,14998
Positive (1),0.7767,0.7571,0.7667,14995



Table 4 - Confusion matrix


,Predicted Negative,Predicted Positive
Actual Negative,11734,3264
Actual Positive,3643,11352


{'Model': '4. Learned embedding + LSTM',
 'Accuracy': 0.7697129330177042,
 'Precision': 0.7766830870279147,
 'Recall': 0.7570523507835946,
 'F1': 0.7667420890885144}

## Pretrained embedding utility

In [19]:
def download_file(url, destination):
    destination = Path(destination)
    if not destination.exists():
        print(f"Downloading {destination.name} (this may take a while)...")
        urllib.request.urlretrieve(url, destination)
    return destination

def build_embedding_matrix(vector_file, dimension, has_header=False, zip_member=None):
    """Read vectors only for words in our vocabulary to limit memory usage."""
    vocab = sequence_vectorizer.get_vocabulary()
    needed = set(vocab)
    matrix = np.random.normal(0, 0.05, size=(len(vocab), dimension)).astype("float32")
    matrix[0] = 0.0
    found = 0
    if str(vector_file).endswith(".zip"):
        archive = zipfile.ZipFile(vector_file)
        stream = archive.open(zip_member) if zip_member else archive.open(archive.namelist()[0])
        import io
        reader = io.TextIOWrapper(stream, encoding="utf-8", errors="ignore")
    else:
        reader = open(vector_file, encoding="utf-8", errors="ignore")
    with reader as f:
        if has_header:
            next(f, None)
        for line in f:
            parts = line.rstrip().split(" ")
            if len(parts) != dimension + 1:
                continue
            word = parts[0]
            if word in needed:
                try:
                    matrix[vocab.index(word)] = np.asarray(parts[1:], dtype="float32")
                    found += 1
                except ValueError:
                    pass
    print(f"Matched pretrained vectors: {found:,}/{len(vocab):,}")
    return matrix

def run_pretrained_lstm(matrix):
    model = build_lstm_model(embedding_matrix=matrix, trainable=False)
    early_stop = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
    model.fit(sequence_train, validation_data=sequence_val, epochs=EPOCHS, callbacks=early_stop, verbose=1)
    return model

## Experiment 5 — GloVe 6B (100-dimensional) + LSTM

In [20]:
glove_zip = download_file(
    "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip", "glove.6B.zip"
)
with zipfile.ZipFile(glove_zip) as zf:
    member = "glove.6B.100d.txt"
    if not Path(member).exists():
        zf.extract(member, ".")
glove_matrix = build_embedding_matrix("glove.6B.100d.txt", dimension=100)
model_5 = run_pretrained_lstm(glove_matrix)

Matched pretrained vectors: 8,800/10,000
Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 57s 246ms/step - accuracy: 0.6791 - loss: 0.5984 - val_accuracy: 0.7200 - val_loss: 0.5575
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 82s 246ms/step - accuracy: 0.7258 - loss: 0.5471 - val_accuracy: 0.7294 - val_loss: 0.5360
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 82s 245ms/step - accuracy: 0.7380 - loss: 0.5279 - val_accuracy: 0.7420 - val_loss: 0.5215
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 81s 239ms/step - accuracy: 0.7472 - loss: 0.5130 - val_accuracy: 0.7447 - val_loss: 0.5170
Epoch 5/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 53s 242ms/step - accuracy: 0.7566 - loss: 0.5016 - val_accuracy: 0.7541 - val_loss: 0.5017


### Experiment 5 — evaluation tables

In [21]:
evaluate_model(model_5, sequence_test, "5. GloVe 6B + LSTM")

RESULTS: 5. GloVe 6B + LSTM

Table 1 - Model summary


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 140)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pretrained_embeddi… │ (None, 140, 100)  │  1,000,000 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 140)       │          0 │ input_layer_4[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     17,024 │ pretrained_embed… │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 32)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         33 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,051,173 (4.01 MB)

 Trainable params: 17,057 (66.63 KB)

...


Table 3 - Per-class metrics


,Precision,Recall,F1,Support
Negative (0),0.7680,0.7219,0.7442,14998
Positive (1),0.7376,0.7819,0.7591,14995



Table 4 - Confusion matrix


,Predicted Negative,Predicted Positive
Actual Negative,10827,4171
Actual Positive,3271,11724


{'Model': '5. GloVe 6B + LSTM',
 'Accuracy': 0.7518754376021072,
 'Precision': 0.7375904372444165,
 'Recall': 0.7818606202067355,
 'F1': 0.7590806086112011}

## Experiment 6 — GloVe-Twitter (100-dimensional) + LSTM

In [22]:
twitter_zip = download_file(
    "https://nlp.stanford.edu/data/glove.twitter.27B.zip", "glove.twitter.27B.zip"
)
with zipfile.ZipFile(twitter_zip) as zf:
    member = "glove.twitter.27B.100d.txt"
    if not Path(member).exists():
        zf.extract(member, ".")
twitter_matrix = build_embedding_matrix("glove.twitter.27B.100d.txt", dimension=100)
model_6 = run_pretrained_lstm(twitter_matrix)

Matched pretrained vectors: 9,089/10,000
Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 56s 246ms/step - accuracy: 0.7122 - loss: 0.5586 - val_accuracy: 0.7518 - val_loss: 0.5091
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 54s 245ms/step - accuracy: 0.7557 - loss: 0.5033 - val_accuracy: 0.7604 - val_loss: 0.4912
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 55s 251ms/step - accuracy: 0.7643 - loss: 0.4879 - val_accuracy: 0.7658 - val_loss: 0.4819
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 55s 251ms/step - accuracy: 0.7697 - loss: 0.4783 - val_accuracy: 0.7712 - val_loss: 0.4748
Epoch 5/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 54s 245ms/step - accuracy: 0.7767 - loss: 0.4708 - val_accuracy: 0.7724 - val_loss: 0.4701


### Experiment 6 — evaluation tables

In [23]:
evaluate_model(model_6, sequence_test, "6. GloVe-Twitter + LSTM")

RESULTS: 6. GloVe-Twitter + LSTM

Table 1 - Model summary


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 140)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pretrained_embeddi… │ (None, 140, 100)  │  1,000,000 │ input_layer_5[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 140)       │          0 │ input_layer_5[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 32)        │     17,024 │ pretrained_embed… │
│                     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 32)        │          0 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 1)         │         33 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,051,173 (4.01 MB)

 Trainable params: 17,057 (66.63 KB)

...


Table 3 - Per-class metrics


,Precision,Recall,F1,Support
Negative (0),0.7688,0.7772,0.7730,14998
Positive (1),0.7747,0.7663,0.7704,14995



Table 4 - Confusion matrix


,Predicted Negative,Predicted Positive
Actual Negative,11656,3342
Actual Positive,3505,11490


{'Model': '6. GloVe-Twitter + LSTM',
 'Accuracy': 0.7717133997932851,
 'Precision': 0.7746763754045307,
 'Recall': 0.7662554184728243,
 'F1': 0.7704428873168606}

## Experiment 7 — FastText English (300-dimensional) + LSTM

In [24]:
fasttext_zip = download_file(
    "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip",
    "wiki-news-300d-1M.vec.zip",
)
with zipfile.ZipFile(fasttext_zip) as zf:
    member = "wiki-news-300d-1M.vec"
    if not Path(member).exists():
        zf.extract(member, ".")
fasttext_matrix = build_embedding_matrix("wiki-news-300d-1M.vec", dimension=300, has_header=True)
model_7 = run_pretrained_lstm(fasttext_matrix)

Matched pretrained vectors: 9,061/10,000
Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 109s 486ms/step - accuracy: 0.7096 - loss: 0.5721 - val_accuracy: 0.7434 - val_loss: 0.5229
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 139s 471ms/step - accuracy: 0.7547 - loss: 0.5104 - val_accuracy: 0.7559 - val_loss: 0.5055
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 91s 416ms/step - accuracy: 0.7630 - loss: 0.4952 - val_accuracy: 0.7599 - val_loss: 0.4931
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 92s 419ms/step - accuracy: 0.7694 - loss: 0.4854 - val_accuracy: 0.7656 - val_loss: 0.4874
Epoch 5/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 152s 467ms/step - accuracy: 0.7742 - loss: 0.4755 - val_accuracy: 0.7677 - val_loss: 0.4834


### Experiment 7 — evaluation tables

In [25]:
evaluate_model(model_7, sequence_test, "7. FastText + LSTM")

RESULTS: 7. FastText + LSTM

Table 1 - Model summary


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 140)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pretrained_embeddi… │ (None, 140, 300)  │  3,000,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 140)       │          0 │ input_layer_6[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 32)        │     42,624 │ pretrained_embed… │
│                     │                   │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 32)        │          0 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,127,973 (11.93 MB)

 Trainable params: 42,657 (166.63 KB)

...


Table 3 - Per-class metrics


,Precision,Recall,F1,Support
Negative (0),0.7579,0.7838,0.7706,14998
Positive (1),0.7761,0.7496,0.7626,14995



Table 4 - Confusion matrix


,Predicted Negative,Predicted Positive
Actual Negative,11755,3243
Actual Positive,3755,11240


{'Model': '7. FastText + LSTM',
 'Accuracy': 0.7666788917414064,
 'Precision': 0.7760823033901816,
 'Recall': 0.7495831943981327,
 'F1': 0.7626026189022321}

## Final test-set comparison

In [26]:
results_df = pd.DataFrame(results).set_index("Model")
results_df = results_df[["Accuracy", "Precision", "Recall", "F1"]].sort_values("F1", ascending=False)
display(results_df.style.format("{:.4f}").highlight_max(color="#d8f3dc"))
print("Top model by test F1:", results_df["F1"].idxmax())
print("Top F1 score:", round(results_df["F1"].max(), 4))

,Accuracy,Precision,Recall,F1
Model,,,,
2. Bigram multi_hot,0.7695,0.7594,0.7889,0.7738
6. GloVe-Twitter + LSTM,0.7717,0.7747,0.7663,0.7704
1. Unigram multi_hot,0.7668,0.7608,0.7783,0.7694
3. Bigram tf_idf,0.7659,0.7594,0.7783,0.7688
4. Learned embedding + LSTM,0.7697,0.7767,0.7571,0.7667
7. FastText + LSTM,0.7667,0.7761,0.7496,0.7626
5. GloVe 6B + LSTM,0.7519,0.7376,0.7819,0.7591


Top model by test F1: 2. Bigram multi_hot
Top F1 score: 0.7738


## Discussion: which model performs best and why?
Best model: Experiment 2 (2-gram multi_hot), with 76.95% accuracy and 77.38% F1. Models 1, 3, 4 and 6 are within about 1 point, so the differences are small.

Why:

Tweets are short, so single words and short phrases carry the sentiment, and bigrams capture them.
The LSTM gives no gain, overfits after epoch 2, and is much slower.
The frozen pretrained embeddings could not adapt to this data. GloVe 6B is worst because it was trained on formal text, while GloVe-Twitter does better because it was trained on tweets.

Conclusion: 2-gram multi_hot is the best choice, being the most accurate, fast and simple. These results come from one run on a 100,000-tweet sample with noisy labels.